# Facility Capacity Training Pipeline — facilities, NFHS-5 enrichment, enriched model

Three clearly delineated parts, in order:

* **Part 1 — Train on facilities only (GPU):** the baseline capacity model from facility-level features alone (numberDoctors, operator/facility type, text signal).
* **Part 2 — NFHS-5 Socioeconomic Enrichment:** select the most socioeconomically representative district indicators (one per correlation cluster) and join them onto facilities via zip → pincode directory → district. Checkpoints the result to `workspace.default.facilities_enriched` (write access verified).
* **Part 3 — Train on the enriched data (GPU):** the same pipeline plus the 15 NFHS district features. Compare its test metrics against Part 1 to see whether they help.

**Setup:** switch the environment (top-right, next to the compute dropdown) from *Standard* to **AI | GPU**, then *Run All*. Parts 1 and 3 train a PyTorch MLP and need the GPU; Part 2 runs anywhere.

Verified on this data: facilities has 10,088 rows, 2,517 with a numeric capacity label; NFHS-5 has 706 districts x 107 numeric indicators; Part 2 picks 15 representatives (max pairwise |r| 0.68); ~60% of labeled facilities match an NFHS district through their zip code.

## Weights & Biases setup

Open this notebook from the project Git folder with `wandb_models.py` and `gpu_capacity_models.py` next to it.
Install the dependencies below. Each training section logs separate PyTorch MLP and
histogram boosting runs with test metrics and the exact fitted pipelines or weights.
The MLP run also records per-epoch training and calibration Huber loss, learning rate,
and the chosen checkpoint.
`RANDOM_STATE` seeds the data split, initial model weights, and batch order.
Runs are offline by default. Set `WANDB_MODE=online`, `WANDB_PROJECT`, and
`WANDB_ENTITY` for a shared project, and provide `WANDB_API_KEY` through a secret.


In [ ]:
%pip install "wandb==0.31.2" "skops==0.16.0" "safetensors==0.8.0"


---

## Part 1 — Train on facilities only (GPU)

The baseline: predict `capacity` from facility-level features alone. Strongest verified signals: `numberDoctors` (Spearman 0.47), `operatorTypeId` (public median 300 vs private 80), `facilityTypeId` (hospital 100 / clinic 25), and text-derived features. About 2,517 of 10,088 rows have a usable capacity label; the rest are dropped from training.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
TBL_NFHS = f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators"
TBL_PINCODE = f"{CATALOG}.{SCHEMA}.india_post_pincode_directory"

TARGET = "capacity"
RANDOM_STATE = 42

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
]
CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [0]:
# Load the facilities table and normalize the messy string columns.
# Many columns are strings holding numbers or literal 'null'/'None' text, so we
# convert them to proper NaN/numeric before anything else.
NULL_STRINGS = {"null", "none", "nan", "n/a", "na", "", "-", "unknown"}

def to_na(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower().where(
        lambda s: ~s.isin(NULL_STRINGS), other=pd.NA
    )

raw = spark.table(TBL_FACILITIES).toPandas()

facilities = raw.copy()
for col in [TARGET] + NUMERIC_FEATURES:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce").astype("float64")
for col in CATEGORICAL_FEATURES:
    category = to_na(facilities[col])
    facilities[col] = category.astype("object").where(category.notna(), np.nan)

# Free-text signal (verified by screening on the 2,517 labeled rows):
#   equipment/capability/procedure/specialties string lengths correlate with capacity
#   (Spearman ~0.33 / 0.38 / 0.23 / 0.15); affiliationTypeIds presence is strong
#   (median 200 when present vs 52 when absent) though only ~1,249 rows have it.
#   description length showed no signal (~0.02) and is left out.
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0).astype("float64")
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

# Keep only rows with a usable target label.
labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"facilities rows: {len(facilities)} | rows with a capacity label: {len(labeled)}")
print(f"Dropped {len(facilities) - len(labeled)} rows without a capacity value")
display(labeled[TARGET].describe().to_frame())

In [0]:
# EDA: target distribution and feature/target relationships.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(labeled[TARGET], bins=50)
axes[0].set_title("Capacity distribution (right-skewed -> log1p helps)")
axes[0].set_xlabel("capacity")

axes[1].hist(np.log1p(labeled[TARGET]), bins=50)
axes[1].set_title("log1p(capacity)")
plt.tight_layout()
plt.show()

corr = labeled[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
corr.plot(kind="barh", ax=ax)
ax.set_title("Correlation of numeric features with capacity")
plt.tight_layout()
plt.show()

cap_by_type = labeled.groupby("facilityTypeId")[TARGET].agg(["count", "median"])
display(cap_by_type)
print(corr)

In [0]:
# Split: no event/observation timestamp exists and rows are independent
# facilities, so a shuffled random split is appropriate (80/10/10).
from sklearn.model_selection import train_test_split

X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Preprocessing fit on the training split only.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train).astype(np.float32)
X_val_prep = preprocessor.transform(X_val).astype(np.float32)
X_test_prep = preprocessor.transform(X_test).astype(np.float32)
print(f"feature matrix shape after preprocessing: {X_train_prep.shape}")

In [0]:
# Fit a histogram-boosting baseline on the same training features.
from gpu_capacity_models import fit_capacity_histogram

baseline = fit_capacity_histogram(preprocessor, X_train_prep, y_train, seed=RANDOM_STATE)


def regression_report(y_true, y_pred, name):
    error = np.asarray(y_pred) - y_true.to_numpy()
    absolute_error = np.abs(error)
    metrics = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "median_ae": np.median(absolute_error),
        "p90_ae": np.quantile(absolute_error, 0.9),
        "mean_bias": error.mean(),
    }
    print(f"{name}: MAE {metrics['mae']:.2f} | RMSE {metrics['rmse']:.2f} | R2 {metrics['r2']:.3f}")
    return metrics


baseline_calibration_metrics = regression_report(y_val, baseline.predict(X_val), "Histogram boosting calibration")
baseline_metrics = regression_report(y_test, baseline.predict(X_test), "Histogram boosting test")


In [0]:
# PyTorch MLP with Huber loss on log1p(bed capacity).
from gpu_capacity_models import fit_capacity_mlp

mlp_fit = fit_capacity_mlp(
    X_train_prep, y_train, X_val_prep, y_val,
    seed=RANDOM_STATE, max_epochs=300, patience=30,
)
mlp = mlp_fit["model"]
device = mlp_fit["device"]


In [0]:
# Evaluate the selected MLP checkpoint in bed units.
def predict_beds(features):
    mlp.eval()
    with torch.inference_mode():
        matrix = torch.as_tensor(features, dtype=torch.float32, device=device)
        return np.expm1(mlp(matrix).cpu().numpy().ravel())


mlp_calibration_metrics = regression_report(y_val, predict_beds(X_val_prep), "MLP calibration")
mlp_pred = predict_beds(X_test_prep)
mlp_metrics = regression_report(y_test, mlp_pred, "MLP test")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, mlp_pred, s=12, alpha=0.5)
ax.plot([0, y_test.max()], [0, y_test.max()], "r--", linewidth=1)
ax.set_xlabel("actual capacity")
ax.set_ylabel("predicted capacity")
ax.set_title("Test set: predicted vs actual")
plt.tight_layout()
plt.show()


In [0]:
# Log both fitted models and aggregate evaluation in Weights & Biases.
from wandb_models import log_sklearn_capacity, log_torch_capacity

shared_config = {
    "variant": "base",
    "source_tables": [TBL_FACILITIES],
    "text_features": TEXT_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "split_unit": "source_row",
    "split_seed": RANDOM_STATE,
    "n_train": len(X_train),
    "n_validation": len(X_val),
    "n_test": len(X_test),
    "target_field": TARGET,
    "target_unit": "beds",
}

capacity_run = log_torch_capacity(
    name="capacity-mlp-base",
    model=mlp,
    preprocessor=preprocessor,
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    metrics=mlp_metrics,
    baseline_metrics=baseline_metrics,
    config={**shared_config, **mlp_fit["config"],
            "prediction_postprocess": "Apply expm1 once to the MLP output to get beds."},
    history=mlp_fit["history"],
    evaluation_tables={"heldout_by_split": [
        {"split": "calibration", "model": "mlp", **mlp_calibration_metrics},
        {"split": "test", "model": "mlp", **mlp_metrics},
        {"split": "calibration", "model": "histogram_boosting", **baseline_calibration_metrics},
        {"split": "test", "model": "histogram_boosting", **baseline_metrics},
    ]},
)

boosting_estimator = baseline.named_steps["histogram_boosting"].regressor_
baseline_run = log_sklearn_capacity(
    name="capacity-hist-gradient-boosting-base",
    model=baseline,
    metrics=baseline_metrics,
    config={**shared_config,
            "device": "cpu",
            "device_name": "CPU",
            "estimator": "HistGradientBoostingRegressor",
            "estimator_parameters": boosting_estimator.get_params(),
            "boosting_iterations": boosting_estimator.n_iter_,
            "prediction_postprocess": "The fitted pipeline applies expm1 and predict() returns beds."},
    target_transform={"training": "log1p", "prediction": "expm1 within pipeline.predict"},
    evaluation_tables={"heldout_by_split": [
        {"split": "calibration", **baseline_calibration_metrics},
        {"split": "test", **baseline_metrics},
    ]},
)
print("MLP W&B run", capacity_run)
print("Histogram boosting W&B run", baseline_run)


In [0]:
# Part 2 — Setup and data loading (enrichment).
import re

import numpy as np
import pandas as pd

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
TBL_NFHS = f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators"
TBL_PINCODE = f"{CATALOG}.{SCHEMA}.india_post_pincode_directory"

# Where the enriched output is checkpointed (change if you lack write access).
ENRICHED_TABLE = "workspace.default.facilities_enriched"

def to_num(s):
    m = re.match(r"^\s*-?\d+(\.\d+)?\s*$", str(s))
    return float(m.group(0)) if m else np.nan

def to_na(s):
    return s.astype("string").str.strip().str.lower().where(
        lambda x: ~x.isin({"null", "none", "nan", "n/a", "na", "", "-", "unknown"}), other=pd.NA
    )

def norm(s):
    return (s.astype("string").str.strip().str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"[^a-z ]", "", regex=True)
            .str.strip())

nfhs_raw = spark.table(TBL_NFHS).toPandas()
num_cols = [c for c in nfhs_raw.columns if c not in ("district_name", "state_ut")]
nfhs_num = pd.DataFrame({c: nfhs_raw[c].map(to_num) for c in num_cols})
print(f"NFHS-5: {len(nfhs_raw)} districts x {len(num_cols)} indicators parsed numeric")

In [0]:
# Part 2 — Select the most socioeconomically representative indicators.
# Candidate pool: education, household amenities, wealth proxies, demographics,
# women's empowerment, nutrition. Survey-count columns excluded.
POOL = [
    "female_population_age_6_years_and_above_ever_schooled_pct",
    "women_age_15_49_who_are_literate_pct",
    "women_age_15_49_with_10_or_more_years_of_schooling_pct",
    "population_below_age_15_years_pct",
    "sex_ratio_total_f_per_1000_m",
    "hh_electricity_pct",
    "hh_improved_water_pct",
    "hh_use_improved_sanitation_pct",
    "households_using_clean_fuel_for_cooking_pct",
    "households_using_iodized_salt_pct",
    "hh_member_covered_health_insurance_pct",
    "w20_24_married_before_age_18_years_pct",
    "w15_24_who_use_menstrual_hygiene_pct",
    "women_age_15_49_years_whose_bmi_bmi_is_underweight_bmi_lt_1_pct",
    "all_w15_49_who_are_anaemic_pct",
    "institutional_birth_5y_pct",
    "births_attended_by_skilled_hp_5y_10_pct",
    "child_u5_who_are_stunted_height_for_age_18_pct",
]

pool = nfhs_num[POOL].dropna(axis=1, thresh=int(0.8 * len(nfhs_num)))
corr = pool.corr(method="spearman").abs()

# Greedy correlation pruning at |r| > 0.7: repeatedly keep the indicator most
# connected to the rest, drop its redundant partners. One representative per
# cluster -> minimal redundancy (max pairwise |r| among picks was 0.68).
remaining = list(pool.columns)
representatives = []
while remaining:
    best = corr.loc[remaining, remaining].mean().idxmax()
    representatives.append(best)
    remaining = [c for c in remaining if c != best and corr.loc[best, c] <= 0.7]

print(f"{len(representatives)} representative indicators (one per correlation cluster):")
for r in representatives:
    print("  -", r)

In [0]:
# Part 2 — Join facilities to NFHS districts via the pincode directory.
facilities = spark.table(TBL_FACILITIES).toPandas()
facilities["capacity_num"] = pd.to_numeric(to_na(facilities["capacity"]), errors="coerce")

pin = spark.table(TBL_PINCODE).select("pincode", "district", "statename").toPandas()
pin["d"], pin["st"] = norm(pin["district"]), norm(pin["statename"])

# Majority (district, state) per pincode = most frequent among its post offices.
cnt = pin.groupby(["pincode", "d", "st"]).size().rename("n").reset_index()
pin_lookup = (cnt.sort_values("n", ascending=False)
                  .drop_duplicates("pincode")
                  .set_index("pincode")[["d", "st"]])

# NFHS feature frame keyed on normalized (district, state).
nfhs_feats = pd.concat(
    [pd.DataFrame({"d": norm(nfhs_raw["district_name"]), "st": norm(nfhs_raw["state_ut"])}),
     nfhs_num[representatives]],
    axis=1,
)

# facilities zip -> pincode lookup -> (district, state) -> NFHS features.
zips = pd.to_numeric(to_na(facilities["address_zipOrPostcode"]), errors="coerce")
facilities["join_d"] = zips.map(pin_lookup["d"])
facilities["join_st"] = zips.map(pin_lookup["st"])

enriched = facilities.merge(nfhs_feats, how="left",
                            left_on=["join_d", "join_st"], right_on=["d", "st"])

matched = enriched["d"].notna()
labeled = enriched["capacity_num"].notna()
print(f"facilities: {len(enriched)} rows")
print(f"matched to an NFHS district: {matched.sum()} ({matched.mean()*100:.0f}%)")
print(f"of capacity-labeled rows: {(matched & labeled).sum()} of {labeled.sum()} ({(matched & labeled).sum()/labeled.sum()*100:.0f}%)")

In [0]:
# Part 2 — Checkpoint the enriched table so Part 3 (and anything else) can
# reuse it without re-running the join. Unmatched rows keep NULL district
# features; Part 3 imputes them, so no rows are lost.
out = enriched.replace({pd.NA: None}).copy()
for c in representatives:
    out[c] = pd.to_numeric(out[c], errors="coerce")

(spark.createDataFrame(out)
      .write.mode("overwrite")
      .saveAsTable(ENRICHED_TABLE))
print(f"wrote {len(out)} rows to {ENRICHED_TABLE}")
display(spark.table(ENRICHED_TABLE).select("unique_id", "capacity_num", *representatives[:3]).limit(5))

---

## Part 3 — Train on the enriched data (GPU)

Switch the environment (next to the compute dropdown) to **AI | GPU** before running this part — the MLP trains on the GPU. Part 2 runs fine on plain serverless.

Notes:

* Only ~60% of labeled facilities matched an NFHS district; unmatched rows have NULL district features, which the preprocessing imputes with the training median — no rows are dropped.
* District indicators are constant within a district, so they add coarse regional signal. Judge whether they help by comparing this notebook's test metrics against the facilities-only notebook.

In [0]:
# Part 3 — Setup.
import torch

import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

TARGET = "capacity"
RANDOM_STATE = 42

# 15 NFHS-5 district indicators picked in Part 1 (one per correlation cluster).
NFHS_FEATURES = [
    "women_age_15_49_with_10_or_more_years_of_schooling_pct",
    "population_below_age_15_years_pct",
    "hh_use_improved_sanitation_pct",
    "women_age_15_49_years_whose_bmi_bmi_is_underweight_bmi_lt_1_pct",
    "hh_electricity_pct",
    "child_u5_who_are_stunted_height_for_age_18_pct",
    "households_using_clean_fuel_for_cooking_pct",
    "w20_24_married_before_age_18_years_pct",
    "female_population_age_6_years_and_above_ever_schooled_pct",
    "hh_member_covered_health_insurance_pct",
    "hh_improved_water_pct",
    "all_w15_49_who_are_anaemic_pct",
    "sex_ratio_total_f_per_1000_m",
    "households_using_iodized_salt_pct",
    "births_attended_by_skilled_hp_5y_10_pct",
]

NUMERIC_FEATURES = [
    "numberDoctors",
    "yearEstablished",
    "latitude",
    "longitude",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements",
    "post_metrics_post_count",
    "number_of_facts_about_the_organization",
    "distinct_social_media_presence_count",
] + NFHS_FEATURES

CATEGORICAL_FEATURES = [
    "facilityTypeId",
    "operatorTypeId",
    "address_stateOrRegion",
    "organization_type",
    "affiliated_staff_presence",
    "custom_logo_presence",
]

print(f"torch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [0]:
# Part 3 — Load and prepare. Uses the in-memory frame from Part 2 if it
# exists; otherwise falls back to the checkpointed table (run Part 2 first).
if "enriched" in globals():
    raw = enriched.copy()
else:
    raw = spark.table(ENRICHED_TABLE).toPandas()

facilities = raw.copy()
for col in [c for c in NUMERIC_FEATURES if c not in NFHS_FEATURES] + [TARGET]:
    facilities[col] = pd.to_numeric(to_na(facilities[col]), errors="coerce").astype("float64")
for col in CATEGORICAL_FEATURES:
    category = to_na(facilities[col])
    facilities[col] = category.astype("object").where(category.notna(), np.nan)
for col in NFHS_FEATURES:
    facilities[col] = pd.to_numeric(facilities[col], errors="coerce").astype("float64")

# Free-text signal (verified screening): length/presence of these scraped
# columns correlates with capacity (equipment .33, capability .38, procedure
# .23, specialties .15; affiliationTypeIds presence median 200 vs 52).
TEXT_FEATURES = ["equipment", "capability", "procedure", "specialties", "affiliationTypeIds"]
for col in TEXT_FEATURES:
    facilities[f"{col}_len"] = to_na(raw[col]).str.len().fillna(0).astype("float64")
    facilities[f"{col}_present"] = to_na(raw[col]).notna().astype(int)
NUMERIC_FEATURES = NUMERIC_FEATURES + [f"{c}_len" for c in TEXT_FEATURES] + [f"{c}_present" for c in TEXT_FEATURES]

labeled = facilities.dropna(subset=[TARGET]).copy()
print(f"rows: {len(facilities)} | with a capacity label: {len(labeled)}")
print(f"labeled rows with NFHS district features: {labeled[NFHS_FEATURES[0]].notna().sum()}")
display(labeled[TARGET].describe().to_frame())

In [0]:
# Part 3 — EDA: target distribution and feature/target relationships.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(labeled[TARGET], bins=50)
axes[0].set_title("Capacity distribution (right-skewed -> log1p helps)")
axes[0].set_xlabel("capacity")

axes[1].hist(np.log1p(labeled[TARGET]), bins=50)
axes[1].set_title("log1p(capacity)")
plt.tight_layout()
plt.show()

corr = labeled[NUMERIC_FEATURES + [TARGET]].corr(method="spearman")[TARGET].drop(TARGET).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
corr.plot(kind="barh", ax=ax)
ax.set_title("Spearman correlation with capacity (facility + NFHS features)")
plt.tight_layout()
plt.show()

In [0]:
# Part 3 — Split: no event/observation timestamp exists and rows are
# independent facilities, so a shuffled random split is appropriate (80/10/10).
X = labeled[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = labeled[TARGET]

X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_STATE)

print(f"train: {len(X_train)} | val: {len(X_val)} | test: {len(X_test)}")

In [0]:
# Part 3 — Preprocessing fit on the training split only. Missing NFHS
# district features (unmatched facilities) are imputed with the median.
preprocessor = ColumnTransformer(
    [
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=5, sparse_output=False))]), CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)
preprocessor.fit(X_train)

X_train_prep = preprocessor.transform(X_train).astype(np.float32)
X_val_prep = preprocessor.transform(X_val).astype(np.float32)
X_test_prep = preprocessor.transform(X_test).astype(np.float32)
print(f"feature matrix shape after preprocessing: {X_train_prep.shape}")

In [0]:
# Part 3 — Fit a histogram-boosting baseline on the same training features.
from gpu_capacity_models import fit_capacity_histogram

baseline = fit_capacity_histogram(preprocessor, X_train_prep, y_train, seed=RANDOM_STATE)


def regression_report(y_true, y_pred, name):
    error = np.asarray(y_pred) - y_true.to_numpy()
    absolute_error = np.abs(error)
    metrics = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "median_ae": np.median(absolute_error),
        "p90_ae": np.quantile(absolute_error, 0.9),
        "mean_bias": error.mean(),
    }
    print(f"{name}: MAE {metrics['mae']:.2f} | RMSE {metrics['rmse']:.2f} | R2 {metrics['r2']:.3f}")
    return metrics


baseline_calibration_metrics = regression_report(y_val, baseline.predict(X_val), "Histogram boosting calibration")
baseline_metrics = regression_report(y_test, baseline.predict(X_test), "Histogram boosting test")


In [0]:
# Part 3 — PyTorch MLP with Huber loss on log1p(bed capacity).
from gpu_capacity_models import fit_capacity_mlp

mlp_fit = fit_capacity_mlp(
    X_train_prep, y_train, X_val_prep, y_val,
    seed=RANDOM_STATE, max_epochs=300, patience=30,
)
mlp = mlp_fit["model"]
device = mlp_fit["device"]


In [0]:
# Part 3 — Evaluate the selected MLP checkpoint in bed units.
def predict_beds(features):
    mlp.eval()
    with torch.inference_mode():
        matrix = torch.as_tensor(features, dtype=torch.float32, device=device)
        return np.expm1(mlp(matrix).cpu().numpy().ravel())


mlp_calibration_metrics = regression_report(y_val, predict_beds(X_val_prep), "MLP calibration")
mlp_pred = predict_beds(X_test_prep)
mlp_metrics = regression_report(y_test, mlp_pred, "MLP test")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, mlp_pred, s=12, alpha=0.5)
ax.plot([0, y_test.max()], [0, y_test.max()], "r--", linewidth=1)
ax.set_xlabel("actual capacity")
ax.set_ylabel("predicted capacity")
ax.set_title("Test set: predicted vs actual")
plt.tight_layout()
plt.show()


In [0]:
# Log both fitted models and aggregate evaluation in Weights & Biases.
from wandb_models import log_sklearn_capacity, log_torch_capacity

shared_config = {
    "variant": "nfhs-enriched",
    "source_tables": [TBL_FACILITIES, TBL_PINCODE, TBL_NFHS],
    "text_features": TEXT_FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "split_unit": "source_row",
    "split_seed": RANDOM_STATE,
    "n_train": len(X_train),
    "n_validation": len(X_val),
    "n_test": len(X_test),
    "target_field": TARGET,
    "target_unit": "beds",
}

capacity_run = log_torch_capacity(
    name="capacity-mlp-nfhs-enriched",
    model=mlp,
    preprocessor=preprocessor,
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    metrics=mlp_metrics,
    baseline_metrics=baseline_metrics,
    config={**shared_config, **mlp_fit["config"],
            "prediction_postprocess": "Apply expm1 once to the MLP output to get beds."},
    history=mlp_fit["history"],
    evaluation_tables={"heldout_by_split": [
        {"split": "calibration", "model": "mlp", **mlp_calibration_metrics},
        {"split": "test", "model": "mlp", **mlp_metrics},
        {"split": "calibration", "model": "histogram_boosting", **baseline_calibration_metrics},
        {"split": "test", "model": "histogram_boosting", **baseline_metrics},
    ]},
)

boosting_estimator = baseline.named_steps["histogram_boosting"].regressor_
baseline_run = log_sklearn_capacity(
    name="capacity-hist-gradient-boosting-nfhs-enriched",
    model=baseline,
    metrics=baseline_metrics,
    config={**shared_config,
            "device": "cpu",
            "device_name": "CPU",
            "estimator": "HistGradientBoostingRegressor",
            "estimator_parameters": boosting_estimator.get_params(),
            "boosting_iterations": boosting_estimator.n_iter_,
            "prediction_postprocess": "The fitted pipeline applies expm1 and predict() returns beds."},
    target_transform={"training": "log1p", "prediction": "expm1 within pipeline.predict"},
    evaluation_tables={"heldout_by_split": [
        {"split": "calibration", **baseline_calibration_metrics},
        {"split": "test", **baseline_metrics},
    ]},
)
print("MLP W&B run", capacity_run)
print("Histogram boosting W&B run", baseline_run)


---

## Next steps

* Compare Part 3's test metrics against Part 1's facilities-only results to see whether the NFHS district features actually help (they only cover ~60% of labeled rows).
* If they help, lower the |r| threshold in Part 2 (0.7 → 0.6) to pull in more clusters of indicators.
* Compare runs in Weights & Biases and use a versioned model artifact for inference.